In [1]:
#Cell 1 – Chia 80/20 train/val
from pathlib import Path
import random
import shutil

# === CHỈNH LẠI ĐƯỜNG DẪN NÀY CHO ĐÚNG MÁY ÔNG ===
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")

IMG_ROOT = ROOT / "images"
TRAIN_DIR = IMG_ROOT / "train"
VAL_DIR   = IMG_ROOT / "val"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

# Lấy ảnh NẰM TRỰC TIẾP trong images/ (chưa chia)
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
all_imgs = [
    p for p in IMG_ROOT.iterdir()
    if p.is_file() and p.suffix.lower() in exts
]

if not all_imgs:
    print("Không thấy ảnh trực tiếp trong 'images/' – có thể ông đã chia train/val rồi.")
else:
    print("Tổng ảnh ban đầu:", len(all_imgs))

    random.seed(1337)
    random.shuffle(all_imgs)

    split_idx = int(len(all_imgs) * 0.8)
    train_imgs = all_imgs[:split_idx]
    val_imgs   = all_imgs[split_idx:]

    def move_batch(files, dst_dir):
        for p in files:
            dst = dst_dir / p.name
            print("MOVE:", p, "->", dst)
            shutil.move(str(p), str(dst))

    move_batch(train_imgs, TRAIN_DIR)
    move_batch(val_imgs, VAL_DIR)

    print("Done. Train:", len(list(TRAIN_DIR.glob('*'))),
          "| Val:", len(list(VAL_DIR.glob('*'))))


Không thấy ảnh trực tiếp trong 'images/' – có thể ông đã chia train/val rồi.


In [2]:

from ultralytics import YOLO
from pathlib import Path

# === ĐƯỜNG DẪN GIỐNG CELL 1 ===
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")
IMG_TRAIN = ROOT / "images" / "train"
IMG_VAL   = ROOT / "images" / "val"

LBL_TRAIN = ROOT / "labels" / "train"
LBL_VAL   = ROOT / "labels" / "val"

LBL_TRAIN.mkdir(parents=True, exist_ok=True)
LBL_VAL.mkdir(parents=True, exist_ok=True)

# === CHỌN MODEL YOLO PRETRAIN (COCO) ===
# nếu ông có sẵn yolo11n.pt trong project thì chỉnh path cho đúng
yolo_model = YOLO("yolo11n.pt")   # hoặc "yolo11s.pt"
print("Loaded YOLO:", yolo_model.model.__class__.__name__)

# Chỉ giữ các class trông giống đồ ăn / chén / dĩa / ly...
ALLOWED = {
    "bowl", "cup", "wine glass", "bottle",
    "fork", "knife", "spoon",
    "pizza", "cake", "sandwich", "hot dog",
    "dining table"
}

exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

def auto_label_split(split_name, img_dir: Path, lbl_dir: Path):
    img_files = []
    for e in exts:
        img_files.extend(img_dir.rglob(e))

    print(f"\n[{split_name}] Số ảnh:", len(img_files))

    for img_path in sorted(img_files):
        label_path = lbl_dir / (img_path.stem + ".txt")

        # Nếu đã có label rồi thì bỏ qua (cho dễ rerun)
        if label_path.exists():
            # print("SKIP (đã có label):", img_path.name)
            continue

        results = yolo_model.predict(
            source=str(img_path),
            conf=0.3,
            verbose=False,
        )

        lines = []

        for r in results:
            if r.boxes is None:
                continue

            h, w = r.orig_img.shape[:2]

            for b in r.boxes:
                cls_idx = int(b.cls.item())
                cls_name = r.names[cls_idx]

                # Lọc class COCO
                if ALLOWED and cls_name not in ALLOWED:
                    continue

                x1, y1, x2, y2 = b.xyxy[0].cpu().numpy().tolist()

                # đổi sang xywh normalized
                xc = (x1 + x2) / 2.0 / w
                yc = (y1 + y2) / 2.0 / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h

                # class duy nhất: 0 (dish)
                lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        if lines:
            label_path.write_text("\n".join(lines), encoding="utf-8")
        else:
            # không detect gì – vẫn ghi file rỗng để dễ kiểm tra
            label_path.write_text("", encoding="utf-8")

        print(f"{img_path.name:30s} -> {label_path.name} ({len(lines)} box)")

# Chạy auto-label cho train & val
auto_label_split("train", IMG_TRAIN, LBL_TRAIN)
auto_label_split("val",   IMG_VAL,   LBL_VAL)

print("\n✅ Xong auto-label. Cấu trúc hiện tại:")
print(" -", IMG_TRAIN)
print(" -", IMG_VAL)
print(" -", LBL_TRAIN)
print(" -", LBL_VAL)


Loaded YOLO: DetectionModel

[train] Số ảnh: 302

[val] Số ảnh: 76

✅ Xong auto-label. Cấu trúc hiện tại:
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/images/train
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/images/val
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/labels/train
 - /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/labels/val


In [3]:
from pathlib import Path

# ĐƯỜNG DẪN GIỐNG NHƯ CÁC CELL TRƯỚC
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")

yaml_path = ROOT / "data_food_plate.yaml"

yaml_text = f"""path: {ROOT}
train: images/train
val: images/val

names:
  0: dish
"""

yaml_path.write_text(yaml_text, encoding="utf-8")
print("Đã tạo file cấu hình:", yaml_path)
print("Nội dung:")
print("--------------------------------")
print(yaml_text)
print("--------------------------------")


Đã tạo file cấu hình: /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/data_food_plate.yaml
Nội dung:
--------------------------------
path: /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables
train: images/train
val: images/val

names:
  0: dish

--------------------------------


In [4]:
from pathlib import Path
from ultralytics import YOLO

# Thư mục đang chứa file yolo.ipynb
NB_DIR = Path.cwd()                     # .../Jupyter/evaluate
print("NB_DIR:", NB_DIR)

# Thư mục để YOLO lưu run (để gọn cứ để ngay trong evaluate luôn)
YOLO_PROJECT = NB_DIR                   # hoặc NB_DIR / "runs_yolo"

ROOT_FOODTABLES = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")
DATA_YAML = ROOT_FOODTABLES / "data_food_plate.yaml"

model = YOLO("yolo11s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project=str(YOLO_PROJECT),          # 👈 lưu run cùng thư mục với yolo.ipynb
    name="MTL_FOOD_PLATE_01",          # thư mục con của run
)

print("✅ Train xong.")
# KHÔNG dùng results.best nữa, tự build path
RUN_DIR = YOLO_PROJECT / "MTL_FOOD_PLATE_01"
YOLO_BEST = RUN_DIR / "weights" / "best.pt"
print("Best weights nằm ở:", YOLO_BEST)


NB_DIR: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo
Ultralytics 8.3.229 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (Quadro RTX 5000, 15925MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/data_food_plate.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=MTL_FO

In [5]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Đường dẫn weight YOLO mới
YOLO_BEST = Path("MTL_FOOD_PLATE_01/weights/best.pt")

det_model = YOLO(str(YOLO_BEST))
print("Loaded:", YOLO_BEST)

# Chọn 1 ảnh bất kỳ trong val để test
ROOT = Path("/media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables")
VAL_IMG_DIR = ROOT / "images" / "val"

val_imgs = sorted([p for p in VAL_IMG_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
print("Số ảnh val:", len(val_imgs))

test_img = val_imgs[0]
print("Test image:", test_img)

# Predict với YOLO mới
results = det_model.predict(
    source=str(test_img),
    conf=0.4,
    verbose=False,
)

# Ultralytics có hàm vẽ sẵn
res = results[0]
plot = res.plot()   # numpy array BGR

# show bằng matplotlib
img_show = Image.fromarray(plot[..., ::-1])  # BGR -> RGB
plt.figure(figsize=(8, 8))
plt.imshow(img_show)
plt.axis("off")
plt.show()


Loaded: MTL_FOOD_PLATE_01/weights/best.pt
Số ảnh val: 76
Test image: /media/mtl/DATA 6TB/AI DATASET/vietnamese-foods/FoodTables/images/val/1669946305-com-44-16697116830802086116054-width2000height2024.jpeg


<Figure size 800x800 with 1 Axes>

In [6]:
from ultralytics import YOLO
from pathlib import Path

YOLO_WEIGHTS = Path("MTL_FOOD_PLATE_01/weights/best.pt")
yolo_model = YOLO(str(YOLO_WEIGHTS))
print("✅ Loaded plate detector:", YOLO_WEIGHTS)


✅ Loaded plate detector: MTL_FOOD_PLATE_01/weights/best.pt


In [7]:
from PIL import Image

def detect_food_boxes(image_path, conf_thres=0.4):
    """
    Dùng YOLO plate detector để tìm các dĩa/tô trên bàn.
    Trả về:
      - img (PIL.Image)
      - boxes: list (x1, y1, x2, y2, cls_id, score)
    """
    img = Image.open(image_path).convert("RGB")

    results = yolo_model.predict(
        source=str(image_path),
        conf=conf_thres,
        verbose=False,
    )

    boxes = []
    for r in results:
        h, w = r.orig_img.shape[:2]
        if r.boxes is None:
            continue

        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxy[0].cpu().numpy().tolist()
            score = float(b.conf.item())
            cls_id = int(b.cls.item())  # YOLO plate: luôn là 0 (dish)

            boxes.append((x1, y1, x2, y2, cls_id, score))

    print(f"YOLO phát hiện {len(boxes)} dĩa/tô (conf ≥ {conf_thres}).")
    return img, boxes


In [8]:
from pathlib import Path
import sys
import json

import torch
from torchvision import transforms

# Nếu chưa có ROOT_DIR thì chắc chắn lại:
# - Notebook trong Jupyter/ → ROOT_DIR = Path.cwd()
# - Notebook trong Jupyter/evaluate/ → ROOT_DIR = parent
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "Yolo":
    ROOT_DIR = NOTEBOOK_DIR.parent
else:
    ROOT_DIR = NOTEBOOK_DIR

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("ROOT_DIR    :", ROOT_DIR)

# Thêm models vào sys.path
sys.path.append(str(ROOT_DIR / "models"))

from efficientnet_b0 import (
    build_model,
    IMAGENET_MEAN,
    IMAGENET_STD,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ====== CHỌN CHECKPOINT CẦN DÙNG ======
# 👉 chỉnh đúng run & tên file ở đây
RUN_NAME = "MTL_TGFOOD"          # ví dụ: run incremental 34 lớp
CKPT_FILE = "mtl_effcientnet_b0_best.pt"                  # hoặc "mtl_effb0_best.pt" với model 33 lớp

CKPT_PATH = ROOT_DIR / "runs" / RUN_NAME / "checkpoints" / CKPT_FILE
RUN_DIR   = CKPT_PATH.parent.parent

print("Classifier ckpt:", CKPT_PATH)
print("Run dir        :", RUN_DIR)

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy checkpoint: {CKPT_PATH}")

# ====== LOAD CHECKPOINT ======
ckpt = torch.load(CKPT_PATH, map_location=device)

# 1) ưu tiên class_names trong checkpoint
class_names = ckpt.get("class_names", None)

# 2) nếu không có thì thử đọc classes.txt trong run
if class_names is None:
    classes_txt = RUN_DIR / "classes.txt"
    if classes_txt.exists():
        print("📄 Đọc class_names từ:", classes_txt)
        lines = classes_txt.read_text(encoding="utf-8").splitlines()
        class_names = [ln.strip() for ln in lines if ln.strip()]

# 3) nếu vẫn chưa có thì fallback sang runs_meta/class_names.json (global)
if class_names is None:
    runs_meta = ROOT_DIR.parent / "runs_meta" / "class_names.json"
    print("⚠️ Không tìm thấy class_names trong ckpt/run → fallback:", runs_meta)
    with open(runs_meta, "r", encoding="utf-8") as f:
        class_names = json.load(f)

num_classes = len(class_names)
print("Số class:", num_classes)
print("Một vài class đầu:", class_names[:10])

# ====== BUILD MODEL ĐÚNG SỐ LỚP & LOAD WEIGHTS ======
model = build_model(
    num_classes=num_classes,
    dropout=0.4,
    pretrained=False,      # dùng weight từ ckpt nên không cần ImageNet
    freeze_backbone=False, # chỉ để build, inference không ảnh hưởng
    device=device,
)

# Nếu checkpoint lưu kiểu {'model_state': ..., ...}
state = ckpt.get("model_state", ckpt)
model.load_state_dict(state)
model.eval()

print("✅ Loaded classifier model.")

# ====== TRANSFORM EVAL (KHÔNG AUGMENT) ======
to_rgb = transforms.Lambda(lambda im: im.convert("RGB"))
eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    to_rgb,
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


NOTEBOOK_DIR: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/Yolo
ROOT_DIR    : /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter
Device: cuda
Classifier ckpt: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/runs/MTL_TGFOOD/checkpoints/mtl_effcientnet_b0_best.pt
Run dir        : /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/runs/MTL_TGFOOD
Số class: 33
Một vài class đầu: ['Banh beo', 'Banh bot loc', 'Banh can', 'Banh canh', 'Banh chung', 'Banh cuon', 'Banh duc', 'Banh gio', 'Banh khot', 'Banh mi']
✅ torch.compile enabled
✅ Loaded classifier model.


In [9]:
import torch
from torchvision import transforms

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval().to(device)

# transform infer = y chang eval_tfms trong efficientnet_b0.py
infer_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda im: im.convert("RGB")),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

@torch.no_grad()
def crop_and_classify(img, boxes, prob_thres=0.5, expand_scale=1.2):
    """
    img   : PIL Image gốc (bàn ăn)
    boxes : list (x1, y1, x2, y2, cls_id, score) từ YOLO
    """
    w, h = img.size
    results = []
    counts = {}

    for (x1, y1, x2, y2, _, det_conf) in boxes:
        # nới box một chút quanh dĩa
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        bw = (x2 - x1) * expand_scale
        bh = (y2 - y1) * expand_scale

        nx1 = max(0, cx - bw / 2)
        ny1 = max(0, cy - bh / 2)
        nx2 = min(w, cx + bw / 2)
        ny2 = min(h, cy + bh / 2)

        crop = img.crop((nx1, ny1, nx2, ny2))
        tensor = infer_tfms(crop).unsqueeze(0).to(device)

        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_prob, pred_idx = torch.max(probs, dim=0)

        pred_prob = float(pred_prob.item())
        pred_idx = int(pred_idx.item())
        pred_name = class_names[pred_idx]   # list class_names load từ runs_meta

        if pred_prob < prob_thres:
            continue  # bỏ những box classifier không chắc

        results.append({
            "box": (nx1, ny1, nx2, ny2),
            "det_conf": float(det_conf),
            "pred_idx": pred_idx,
            "pred_name": pred_name,
            "pred_prob": pred_prob,
        })

        counts[pred_name] = counts.get(pred_name, 0) + 1

    return results, counts


In [10]:
from PIL import ImageDraw, ImageFont
import matplotlib.pyplot as plt

def visualize_results(img, results, font_size=18, save_path=None):
    """
    img      : PIL image gốc
    results  : list dict { "box": (x1,y1,x2,y2), "pred_name", "pred_prob", ... }
    """
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        font = ImageFont.load_default()

    for r in results:
        x1, y1, x2, y2 = r["box"]
        name = r.get("pred_name", "?")
        prob = r.get("pred_prob", 0.0)

        label = f"{name} ({prob:.2f})"

        # --- tính width/height của text ---
        try:
            # Pillow cũ
            tw, th = draw.textsize(label, font=font)
        except AttributeError:
            # Pillow mới (>=10) không còn textsize
            bbox = draw.textbbox((0, 0), label, font=font)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]
        # ----------------------------------

        # vẽ khung
        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)

        # vẽ label nền vàng
        yy1 = max(0, y1 - th - 4)
        draw.rectangle([x1, yy1, x1 + tw + 4, yy1 + th + 4], fill="yellow")
        draw.text((x1 + 2, yy1 + 2), label, font=font, fill="black")

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        img.save(save_path)
        print("💾 Đã lưu ảnh có box tại:", save_path)

    # show trong notebook
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


In [11]:
def print_summary_counts(counts):
    """
    counts: dict {class_name: count}
    In tổng kết số món trên bàn theo class.
    """
    if not counts:
        print("👉 Không có món nào được classifier nhận ra (sau khi lọc prob).")
        return

    total = sum(counts.values())
    print("👉 Bàn ăn có tổng cộng:", total, "món (theo classifier).")
    print()
    for name, c in sorted(counts.items(), key=lambda x: x[0]):
        print(f"  - {name}: {c}")


In [12]:
import matplotlib.pyplot as plt

def debug_show_crops(img, results, max_show=10):
    n = min(len(results), max_show)
    if n == 0:
        print("Không có kết quả nào trong results để debug.")
        return

    plt.figure(figsize=(4 * n, 4))

    for i, r in enumerate(results[:n], start=1):
        x1, y1, x2, y2 = r["box"]                    # 👈 lấy đúng key
        name = r.get("pred_name", "?")
        prob = r.get("pred_prob", 0.0)

        crop = img.crop((x1, y1, x2, y2))

        ax = plt.subplot(1, n, i)
        ax.imshow(crop)
        ax.axis("off")
        ax.set_title(f"{name}\n({prob:.2f})")

    plt.tight_layout()
    plt.show()


In [13]:
from pathlib import Path

# 1) Chọn ảnh thật (bàn ăn ngoài đời)

test_image = Path("/home/mtl/Downloads/An-Lac-Tam-1.jpg")  # ✅ Path
# hoặc 1 ảnh bất kỳ ông thích, free thay đường dẫn

print("Test image:", test_image)

# 2) YOLO detect các dĩa/tô
img, boxes = detect_food_boxes(str(test_image), conf_thres=0.4)

# 3) EfficientNet crop + classify
results, counts = crop_and_classify(
    img,
    boxes,
    prob_thres=0.7,     # bỏ bớt box classifier không chắc
    expand_scale=1,   # nới box rộng hơn dĩa
)

# 4) Lưu ảnh có box ra thư mục output
out_dir = ROOT_DIR / "images" / "detect_foods"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{test_image.stem}_det{test_image.suffix}"


debug_show_crops(img, results, max_show=12)
visualize_results(img, results, save_path=out_path)
# 5) In tổng kết số món
print_summary_counts(counts)
print("💾 Đã lưu ảnh có box tại:", out_path)


Test image: /home/mtl/Downloads/An-Lac-Tam-1.jpg
YOLO phát hiện 13 dĩa/tô (conf ≥ 0.4).


W1119 17:57:38.672000 9170 torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode


<Figure size 3200x400 with 8 Axes>

💾 Đã lưu ảnh có box tại: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/images/detect_foods/An-Lac-Tam-1_det.jpg


<Figure size 800x800 with 1 Axes>

👉 Bàn ăn có tổng cộng: 8 món (theo classifier).

  - Banh beo: 1
  - Banh bot loc: 1
  - Banh canh: 2
  - Canh chua: 3
  - Cao lau: 1
💾 Đã lưu ảnh có box tại: /media/mtl/DATA 6TB/PROJECT AI/DL-DuDoan33MonAnVietNam/Jupyter/images/detect_foods/An-Lac-Tam-1_det.jpg
